# Liver Disease Prediction using Machine Learning

This notebook demonstrates the process of building a machine learning model to predict liver disease using the Indian Liver Patient Dataset (ILPD). It covers data loading, preprocessing, model training, and evaluation.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve
from sklearn.pipeline import Pipeline
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
import sys

# Add project root to sys.path
sys.path.append(os.path.abspath('../'))

from src.utils.data_preprocessing import preprocess_tabular_data

# --- 1. Data Loading ---
data_path = "../data/raw/ILPD.csv"
column_names = [
    "Age", "Gender", "Total_Bilirubin", "Direct_Bilirubin", "Alkaline_Phosphotase",
    "Alamine_Aminotransferase", "Aspartate_Aminotransferase", "Total_Protiens",
    "Albumin", "Albumin_and_Globulin_Ratio", "Dataset"
]

try:
    data = pd.read_csv(data_path, names=column_names)
    print("✅ Dataset loaded successfully.")
except FileNotFoundError:
    print(f"⚠️ {data_path} not found. Creating dummy dataset...")
    np.random.seed(42)
    data = pd.DataFrame({
        "Age": np.random.randint(10, 90, 100),
        "Gender": np.random.choice(["Male", "Female"], 100),
        "Total_Bilirubin": np.random.uniform(0.5, 75, 100),
        "Direct_Bilirubin": np.random.uniform(0.1, 20, 100),
        "Alkaline_Phosphotase": np.random.randint(60, 2000, 100),
        "Alamine_Aminotransferase": np.random.randint(10, 500, 100),
        "Aspartate_Aminotransferase": np.random.randint(10, 500, 100),
        "Total_Protiens": np.random.uniform(2.0, 10.0, 100),
        "Albumin": np.random.uniform(1.0, 6.0, 100),
        "Albumin_and_Globulin_Ratio": np.random.uniform(0.3, 2.0, 100),
        "Dataset": np.random.choice([1, 2], 100)
    })

# Convert target column to binary
data["Dataset"] = data["Dataset"].map({2: 0, 1: 1})

# Preprocessing setup
numerical_cols = [
    "Age", "Total_Bilirubin", "Direct_Bilirubin", "Alkaline_Phosphotase",
    "Alamine_Aminotransferase", "Aspartate_Aminotransferase",
    "Total_Protiens", "Albumin", "Albumin_and_Globulin_Ratio"
]
categorical_cols = ["Gender"]
target_column = "Dataset"

X = data.drop(columns=[target_column])
y = data[target_column]

# Use preprocess_tabular_data to get preprocessor
_, _, preprocessor = preprocess_tabular_data(data, numerical_cols, categorical_cols, target_column)

# Build pipeline (preprocessor + model)
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(solver="liblinear", random_state=42))
])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train pipeline
print("🚀 Training Logistic Regression model...")
model.fit(X_train, y_train)
print("✅ Model training complete.")

# Predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Evaluation
print("\n📊 Model Evaluation Metrics:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Confusion Matrix Plot
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ROC Curve Plot
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_pred_proba):.2f}", color="orange")
plt.plot([0, 1], [0, 1], linestyle="--", color="navy")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# Save as ONE pipeline (model + preprocessor together)
joblib.dump(model, "../models/liver_disease_pipeline.pkl")
print("✅ Pipeline saved as ../models/liver_disease_pipeline.pkl")


ModuleNotFoundError: No module named 'src'